# WaveForge — Brain Haemorrhage Dataset Generator

**Just click Run All — fully automated.**

Generates a medically realistic FDTD simulation dataset for deep learning detection
of intracranial haemorrhage using microwave backscatter signals.

### What this notebook generates

| Property | Value |
|----------|-------|
| Frequency | 1.0 GHz (Cole-Cole tissue properties) |
| Grid | 64³ cells at 3mm/cell (192mm domain) |
| Antennas | 8-element ring, r=30 cells |
| Classes | 0=healthy, 1=epidural, 2=subdural, 3=intracerebral |
| Distribution | 25% each class |
| Blood aging | acute / subacute / chronic |
| Phantom A | training/validation |
| Phantom B | test set (independent geometry) |

### Output per sample (.npz)
- `signals_scattered` (8, 8, 300) — background-subtracted microwave signals
- `das_image` (40, 40) — DAS backprojection image
- `label`, `bleed_type`, `bleed_age`, `bleed_center_mm`, `bleed_radius_mm`
- Full metadata for reproducibility

---
**Accelerator:** GPU → T4 x2  
**Estimated time:** ~8h for 2000 samples on 2×T4

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 1 — Setup: clone repo, verify GPUs                               ║
# ╚══════════════════════════════════════════════════════════════════════════╝
import subprocess, sys, os, pathlib

REPO_URL = 'https://github.com/shahzaibshazoo/waveforge.git'
REPO_DIR = pathlib.Path('/kaggle/working/waveforge')
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=True)

src_path = str(REPO_DIR / 'src')
if src_path not in sys.path: sys.path.insert(0, src_path)
os.chdir(REPO_DIR)

import torch, numpy as np, json, time, datetime
assert torch.cuda.is_available(), 'No GPU — enable T4 x2 accelerator!'
N_GPUS = torch.cuda.device_count()
print(f'GPUs: {N_GPUS}')
for i in range(N_GPUS):
    p = torch.cuda.get_device_properties(i)
    print(f'  [{i}] {p.name} {p.total_memory/1e9:.1f}GB')
print(f'PyTorch: {torch.__version__}')
print('✅ Ready')

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 2 — Configuration                                                 ║
# ╚══════════════════════════════════════════════════════════════════════════╝

# ── Core parameters ──────────────────────────────────────────────────────
FREQ_GHZ       = 1.0     # Centre frequency (Cole-Cole at this freq)
GRID_SIZE      = 64      # N for NxNxN grid (64 = 192mm domain at 3mm/cell)
DX_MM          = 3.0     # Cell size in mm
N_TX           = 8       # Antenna elements
RING_RADIUS    = 30      # Ring radius in cells
N_STEPS        = 300     # Time steps per TX sim

# ── Dataset size ─────────────────────────────────────────────────────────
N_TRAIN        = 1600    # Samples on Phantom A (training)
N_TEST         = 400     # Samples on Phantom B (independent test)
TOTAL          = N_TRAIN + N_TEST

# ── Output ───────────────────────────────────────────────────────────────
OUTPUT_ROOT    = pathlib.Path('/kaggle/working/brain_haemorrhage_dataset')
TRAIN_DIR      = OUTPUT_ROOT / 'phantom_A_train'
TEST_DIR       = OUTPUT_ROOT / 'phantom_B_test'

# ── Seeds ────────────────────────────────────────────────────────────────
TRAIN_SEED     = 0
TEST_SEED      = 999999   # Different seed space for test phantom

# ── GPU assignment ───────────────────────────────────────────────────────
# GPU 0 generates Phantom A (training), GPU 1 generates Phantom B (test)
# If only 1 GPU, both run sequentially on GPU 0
GPU0 = 'cuda:0'
GPU1 = 'cuda:1' if N_GPUS > 1 else 'cuda:0'

print('Configuration:')
print(f'  Grid: {GRID_SIZE}³, dx={DX_MM}mm, domain={GRID_SIZE*DX_MM:.0f}mm')
print(f'  Freq: {FREQ_GHZ} GHz, {N_TX} antennas, {N_STEPS} steps')
print(f'  Train: {N_TRAIN} samples (Phantom A, GPU0={GPU0})')
print(f'  Test:  {N_TEST} samples (Phantom B, GPU1={GPU1})')
print(f'  Total: {TOTAL} samples')

# Estimate time
sec_per_sample = N_TX * N_STEPS * GRID_SIZE**3 / 567e6 * 2  # T4 peak
total_h = sec_per_sample * TOTAL / N_GPUS / 3600
print(f'  Est:  ~{sec_per_sample:.0f}s/sample → ~{total_h:.1f}h total ({N_GPUS} GPUs)')

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 3 — Validate 3 samples before full run                           ║
# ╚══════════════════════════════════════════════════════════════════════════╝
from datasets.generator import BrainDatasetGenerator

print('Running 3 validation samples...')
val_gen = BrainDatasetGenerator(
    output_dir=str(OUTPUT_ROOT / 'validation'),
    freq_hz=FREQ_GHZ * 1e9,
    grid_size=GRID_SIZE,
    dx_mm=DX_MM,
    n_tx=N_TX,
    ring_radius_cells=RING_RADIUS,
    n_steps=N_STEPS,
    device=GPU0,
)

val_manifest = val_gen.generate_balanced_dataset(
    n_samples=4,  # one per class
    phantom_id='A',
    base_seed=42,
    show_progress=True,
)

# Load and inspect one sample
sample_path = val_manifest['sample_paths'][0]
s = np.load(sample_path, allow_pickle=True)
print(f'\nSample 0:')
print(f'  signals_scattered shape: {s["signals_scattered"].shape}')
print(f'  das_image shape: {s["das_image"].shape}')
print(f'  label: {s["label"]} type: {s["bleed_type"]} age: {s["bleed_age"]}')
print(f'  bleed_radius_mm: {s["bleed_radius_mm"]}')
print(f'  signal_energy: {(s["signals_scattered"]**2).sum():.4e}')
print('\n✅ Validation passed — starting full generation')

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 4 — Generate training set (Phantom A, GPU 0)                      ║
# ╚══════════════════════════════════════════════════════════════════════════╝
print(f'Generating {N_TRAIN} training samples (Phantom A) on {GPU0}...')
print(f'Output: {TRAIN_DIR}')
print()

t0 = time.time()
train_gen = BrainDatasetGenerator(
    output_dir=str(TRAIN_DIR),
    freq_hz=FREQ_GHZ * 1e9,
    grid_size=GRID_SIZE,
    dx_mm=DX_MM,
    n_tx=N_TX,
    ring_radius_cells=RING_RADIUS,
    n_steps=N_STEPS,
    device=GPU0,
    seed=TRAIN_SEED,
)

train_manifest = train_gen.generate_balanced_dataset(
    n_samples=N_TRAIN,
    phantom_id='A',
    base_seed=TRAIN_SEED,
    show_progress=True,
)

train_time = time.time() - t0
print(f'\nTraining set done: {train_time/3600:.2f}h')
print(f'Class dist: {train_manifest["class_counts"]}')

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 5 — Generate test set (Phantom B, GPU 1)                         ║
# ╚══════════════════════════════════════════════════════════════════════════╝
print(f'Generating {N_TEST} test samples (Phantom B, INDEPENDENT) on {GPU1}...')
print('NOTE: Phantom B has different skull thickness — tests generalisation!')
print()

t0 = time.time()
test_gen = BrainDatasetGenerator(
    output_dir=str(TEST_DIR),
    freq_hz=FREQ_GHZ * 1e9,
    grid_size=GRID_SIZE,
    dx_mm=DX_MM,
    n_tx=N_TX,
    ring_radius_cells=RING_RADIUS,
    n_steps=N_STEPS,
    device=GPU1,
    seed=TEST_SEED,
)

test_manifest = test_gen.generate_balanced_dataset(
    n_samples=N_TEST,
    phantom_id='B',
    base_seed=TEST_SEED,
    show_progress=True,
)

test_time = time.time() - t0
print(f'\nTest set done: {test_time/3600:.2f}h')
print(f'Class dist: {test_manifest["class_counts"]}')

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 6 — Build master dataset manifest and validate                   ║
# ╚══════════════════════════════════════════════════════════════════════════╝
import subprocess

# Get waveforge commit
try:
    commit = subprocess.check_output(
        ['git', '-C', str(REPO_DIR), 'rev-parse', '--short', 'HEAD'],
        text=True
    ).strip()
except Exception:
    commit = 'unknown'

# Merge train + test indices (offset test indices)
n_train_ok = train_manifest['n_completed']
n_test_ok  = test_manifest['n_completed']

master_manifest = {
    'version': '1.0',
    'created_at': datetime.datetime.now().isoformat(),
    'waveforge_commit': commit,

    # Dataset stats
    'n_total': n_train_ok + n_test_ok,
    'n_train': n_train_ok,
    'n_test': n_test_ok,
    'train_class_counts': train_manifest['class_counts'],
    'test_class_counts': test_manifest['class_counts'],
    'class_names': {0: 'healthy', 1: 'epidural', 2: 'subdural', 3: 'intracerebral'},

    # Physics
    'freq_hz': FREQ_GHZ * 1e9,
    'grid_shape': [GRID_SIZE, GRID_SIZE, GRID_SIZE],
    'dx_mm': DX_MM,
    'n_tx': N_TX,
    'n_rx': N_TX,
    'n_steps': N_STEPS,

    # Paths
    'train_dir': str(TRAIN_DIR),
    'test_dir': str(TEST_DIR),
    'train_labels': train_manifest['labels'],
    'test_labels': test_manifest['labels'],

    # Phantom info
    'train_phantom': 'A',
    'test_phantom': 'B',
    'train_seed': TRAIN_SEED,
    'test_seed': TEST_SEED,
}

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
manifest_path = OUTPUT_ROOT / 'dataset_manifest.json'
with open(manifest_path, 'w') as f:
    json.dump(master_manifest, f, indent=2)

print('=== DATASET SUMMARY ===')
print(f'  Total:    {n_train_ok + n_test_ok} samples')
print(f'  Train:    {n_train_ok} (Phantom A)')
print(f'  Test:     {n_test_ok} (Phantom B — independent)')
print(f'  Commit:   {commit}')
print(f'  Manifest: {manifest_path}')

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 7 — Visualise sample signals and DAS images                      ║
# ╚══════════════════════════════════════════════════════════════════════════╝
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt

# Load one sample from each class
label_to_name = {0:'Healthy', 1:'Epidural', 2:'Subdural', 3:'Intracerebral'}
samples_by_label = {}
for path, label in zip(train_manifest['sample_paths'], train_manifest['labels']):
    if label not in samples_by_label:
        samples_by_label[label] = np.load(path, allow_pickle=True)
    if len(samples_by_label) == 4:
        break

fig, axes = plt.subplots(4, 2, figsize=(14, 16))
fig.suptitle('WaveForge Brain Dataset — One Sample per Class', fontsize=14, fontweight='bold')

for row, label in enumerate(sorted(samples_by_label)):
    s = samples_by_label[label]
    name = label_to_name[label]
    age  = str(s['bleed_age']) if str(s['bleed_age']) != 'none' else ''
    r_mm = float(s['bleed_radius_mm'])

    # Left: TX0→all RX scattered signals
    ax = axes[row, 0]
    scat = s['signals_scattered']  # (8, 8, N_steps)
    t_ns = np.arange(scat.shape[2]) * float(s['dt_s']) * 1e9
    for rx in range(scat.shape[1]):
        ax.plot(t_ns, scat[0, rx], alpha=0.6, lw=0.8)
    title = f'{name}'
    if r_mm > 0:
        title += f' ({age}, r={r_mm:.0f}mm)'
    ax.set(title=title, xlabel='Time (ns)', ylabel='Ez (V/m)')
    ax.grid(alpha=0.3)

    # Right: DAS image
    ax = axes[row, 1]
    das = s['das_image']
    im = ax.imshow(das, cmap='hot', origin='lower',
                   extent=[0, GRID_SIZE*DX_MM, 0, GRID_SIZE*DX_MM])
    ax.set(title=f'DAS backprojection', xlabel='x (mm)', ylabel='y (mm)')
    if r_mm > 0:
        cx = float(s['bleed_center_mm'][0])
        cy = float(s['bleed_center_mm'][1])
        ax.plot(cx, cy, 'c+', markersize=15, mew=2, label='true pos')
        ax.legend(fontsize=8)
    plt.colorbar(im, ax=ax)

plt.tight_layout()
os.makedirs('docs/assets', exist_ok=True)
fig.savefig('docs/assets/brain_dataset_samples.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → docs/assets/brain_dataset_samples.png')

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 8 — Validate all 5 dataset quality rules                         ║
# ╚══════════════════════════════════════════════════════════════════════════╝
import math

print('=== Dataset Quality Validation ===')
print()

# Rule 1: Frequency-dependent tissue properties
from datasets.brain.tissue_library import tissue_at_freq
gm_1, _ = tissue_at_freq('gray_matter', FREQ_GHZ * 1e9)
gm_2, _ = tissue_at_freq('gray_matter', 2.4e9)
r1_ok = abs(gm_1 - gm_2) > 1.0
print(f'Rule 1 (Cole-Cole): GM eps_r = {gm_1:.1f} @ {FREQ_GHZ}GHz vs {gm_2:.1f} @ 2.4GHz  → {"✓" if r1_ok else "✗"}')

# Rule 2: No bleeds in bone
from datasets.brain.phantom import PHANTOM_A
bone_violations = 0
for path, label in zip(train_manifest['sample_paths'][:100], train_manifest['labels'][:100]):
    if label == 0: continue
    s = np.load(path, allow_pickle=True)
    centre = s['bleed_center_cells']
    g = PHANTOM_A
    cx = cy = cz = GRID_SIZE // 2
    d = math.sqrt(sum((int(centre[i]) - [cx,cy,cz][i])**2 for i in range(3)))
    if d > g.skull_inner_r + 3:  # allow small margin for int rounding
        bone_violations += 1
r2_ok = bone_violations == 0
print(f'Rule 2 (Anatomy):  Bone violations = {bone_violations}/100 samples  → {"✓" if r2_ok else "✗"}')

# Rule 3: Blood aging
ages = train_manifest['bleed_ages']
bleed_ages = [a for a in ages if a != 'none']
unique_ages = set(bleed_ages)
r3_ok = len(unique_ages) >= 2
print(f'Rule 3 (Aging):    Age types present = {unique_ages}  → {"✓" if r3_ok else "✗"}')

# Rule 4a: Class balance
counts = train_manifest['class_counts']
total_c = sum(counts.values())
fracs = {k: v/total_c for k,v in counts.items()}
r4a_ok = all(0.15 <= f <= 0.35 for f in fracs.values())
print(f'Rule 4a (Balance): class fractions = {fracs}  → {"✓" if r4a_ok else "✗"}')

# Rule 4b: Independent test phantom
from datasets.brain.phantom import PHANTOM_A, PHANTOM_B
r4b_ok = PHANTOM_A.skull_inner_r != PHANTOM_B.skull_inner_r
print(f'Rule 4b (Phantom): A skull_inner_r={PHANTOM_A.skull_inner_r}, B={PHANTOM_B.skull_inner_r}  → {"✓" if r4b_ok else "✗"}')

all_ok = r1_ok and r2_ok and r3_ok and r4a_ok and r4b_ok
print()
print(f'{"✅ ALL 5 RULES PASS — dataset is publication-ready" if all_ok else "❌ Some rules failed — review above"}')

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 9 — Package outputs for download                                 ║
# ╚══════════════════════════════════════════════════════════════════════════╝
import shutil

OUT = pathlib.Path('/kaggle/working/waveforge_brain_outputs')
OUT.mkdir(exist_ok=True)

# Copy manifest
shutil.copy(manifest_path, OUT)

# Copy visualisation
vis = pathlib.Path('docs/assets/brain_dataset_samples.png')
if vis.exists(): shutil.copy(vis, OUT)

# List generated files
train_files = sorted(TRAIN_DIR.glob('*.npz'))
test_files  = sorted(TEST_DIR.glob('*.npz'))

total_size_mb = sum(f.stat().st_size for f in train_files + test_files) / 1e6

print(f'Dataset files:')
print(f'  Train samples: {len(train_files)} files in {TRAIN_DIR}')
print(f'  Test samples:  {len(test_files)} files in {TEST_DIR}')
print(f'  Total size:    {total_size_mb:.1f} MB')
print(f'  Manifest:      {manifest_path}')
print()
print('Files available in Kaggle output panel.')
print('🏁 Dataset generation complete!')